# 10b -- MTMC Stage 3: FAISS Indexing

**Prerequisite**: attach **10a's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-10a-stages-0-2" -> add`

This mounts the checkpoint at `/kaggle/input/mtmc-10a-stages-0-2/`.

| Stage | What | Time |
|---|---|---|
| 3 | Build FAISS similarity index over ReID features | ~1 min |

After this runs, attach **this** notebook's output to **10c**.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _major, _minor = _cap.strip().split(".")
        _sm = int(_major) * 10 + int(_minor)
        if _sm < 70:
            print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) — installing compatible PyTorch ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu124",
            ])
            print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies
(No GPU models needed -- much faster than 10a)

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

try:
    import trackeval; print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("trackeval", "trackeval"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 10a

Finds `checkpoint.tar.gz` in `/kaggle/input/mtmc-10a-stages-0-2/` and extracts to `/tmp/pipeline_run/`.

In [ ]:
PREV_SLUG = "mtmc-10a-stages-0-2"
PREV_INPUT = Path("/kaggle/input") / PREV_SLUG

# Robust path discovery in case Kaggle changes the mount format
if not PREV_INPUT.exists():
    for p in Path("/kaggle/input").iterdir():
        if PREV_SLUG in p.name or "stages012" in p.name or "10a" in p.name:
            PREV_INPUT = p; break

cp = PREV_INPUT / "checkpoint.tar.gz"

# Fallback: download via Kaggle API if not found in mounted input
if not cp.exists():
    print(f"checkpoint.tar.gz not found at {cp} -- attempting API download")
    import subprocess as _sp
    _dl_dir = Path("/tmp/kaggle_dl")
    _dl_dir.mkdir(parents=True, exist_ok=True)
    _r = _sp.run(
        ["kaggle", "kernels", "output",
         "yahiaakhalafallah/mtmc-10a-stages-0-2",
         "--file", "checkpoint.tar.gz",
         "-p", str(_dl_dir)],
        capture_output=True, text=True
    )
    print(_r.stdout); print(_r.stderr)
    cp = _dl_dir / "checkpoint.tar.gz"

assert cp.exists(), f"checkpoint.tar.gz not found at {cp}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {cp.stat().st_size/1024**2:.1f} MB ...")
with tarfile.open(str(cp), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR  = EXTRACT_DIR / RUN_NAME
print(f"\u2713 Checkpoint extracted -- run: {RUN_NAME}")
for s in ["stage1", "stage2"]:
    d = RUN_DIR / s
    if d.exists(): print(f"  {s}: {len(list(d.rglob('*')))} files")

## 3. Run Stage 3 (FAISS Indexing)

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "3",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
]
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"\u2717 FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"\u2713 Stage 3 done in {elapsed/60:.1f} min")

## 4. Save Checkpoint for 10c

In [ ]:
checkpoint_path_out = Path("/kaggle/working/checkpoint.tar.gz")
metadata_path_out   = Path("/kaggle/working/run_metadata.json")
with open(metadata_path_out, "w") as f:
    json.dump({"run_name": RUN_NAME}, f)

with tarfile.open(str(checkpoint_path_out), "w:gz") as tar:
    tar.add(str(metadata_path_out), arcname="run_metadata.json")

    for stage in ["stage1", "stage2", "stage3"]:
        stage_dir = RUN_DIR / stage
        if stage_dir.exists():
            n = 0
            for fpath in stage_dir.rglob("*"):
                if fpath.is_file():
                    tar.add(str(fpath), arcname=f"{RUN_NAME}/{stage}/{fpath.relative_to(stage_dir)}")
                    n += 1
            print(f"  + {stage}/ ({n} files)")

    # Forward GT annotations from 10a's checkpoint
    gt_dir = EXTRACT_DIR / "gt_annotations"
    if gt_dir.exists():
        n = 0
        for fpath in gt_dir.rglob("*"):
            if fpath.is_file():
                tar.add(str(fpath), arcname=f"gt_annotations/{fpath.relative_to(gt_dir)}")
                n += 1
        print(f"  + gt_annotations/ ({n} files forwarded)")

size_mb = checkpoint_path_out.stat().st_size / 1024**2
print(f"\n\u2713 Checkpoint: {checkpoint_path_out}  ({size_mb:.1f} MB)")
print("  Next: attach this notebook's output to 10c, then push 10c.")